# FLAG 2027 - Notebook 02: Clean FOP baseline + honest validation + first submission

**What this notebook does**
1. Loads the pre-extracted features (only the small csv/txt files - fast).
2. Re-implements the official FOP baseline cleanly (works on GPU or CPU).
3. **Honest validation:** 5-fold cross-validation over *people* - the model is always tested on people it never saw, with REAL labels, in both conditions (standard + same-gender). The official code's validation was broken.
4. Trains a final model on all 70 people and writes the **CodaBench submission ZIP** to Drive.

**How to run:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all. About 10-15 minutes.

**Rule we respect:** dev data is only scored by the finished model - nothing is learned from it.

**When done:** paste the **SUMMARY** box to Claude.

In [ ]:
# Cell 1 - Drive + GPU check
from google.colab import drive
drive.mount('/content/drive')
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev, '| GPU:', torch.cuda.get_device_name(0) if dev == 'cuda' else 'none (fine, just slower)')

In [ ]:
# Cell 2 - extract only the small csv/txt files from the zips (skip the raw jpg/wav)
import os, json, time, zipfile
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
P = json.load(open(os.path.join(PROJECT, 'paths.json')))
ZIP_DIR = P['ZIP_DIR']
LOCAL = '/content/flag_small'
os.makedirs(LOCAL, exist_ok=True)

for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if os.path.exists(flag):
        print(z, 'already extracted'); continue
    t0 = time.time()
    with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
        members = [m for m in zf.namelist() if m.endswith(('.csv', '.txt'))]
        zf.extractall(LOCAL, members=members)
    open(flag, 'w').close()
    print('%s: %d small files extracted in %.0f s' % (z, len(members), time.time() - t0))

INDEX = []
for root, dirs, files in os.walk(LOCAL):
    for f in files:
        if f.endswith(('.csv', '.txt')):
            INDEX.append(os.path.join(root, f))

def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]

SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)

In [ ]:
# Cell 3 - load training features, identities and gender
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    X = df.iloc[:, :dim].values.astype(np.float32)
    y = df.iloc[:, dim].values if df.shape[1] > dim else None
    return X, y

Xf, yf = read_feat(find_one('train_English_faces.csv'), 4096)
Xv, yv = read_feat(find_one('train_English_voices.csv'), 192)
assert (yf == yv).all()
y = yf.astype(int)

# speaker name (id028 ...) for each class number, from the train list
lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
row_cls = np.array([int(l[5]) for l in lines])
note('Train list rows aligned with CSV labels: %.0f%%' % (100 * (row_cls == y).mean()))
cls2spk = {int(l[5]): l[4] for l in lines}

# gender meta: two columns, NO header (id001,f)
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
g = np.array([spk2g.get(cls2spk.get(c, '?'), '?') for c in y])

ids = np.unique(y)
id_gender = {c: g[y == c][0] for c in ids}
vals, cnts = np.unique(list(id_gender.values()), return_counts=True)
note('Gender meta rows: %d | identities by gender: %s' % (len(meta), dict(zip(vals, cnts))))
if '?' in id_gender.values():
    note('WARNING: some identities have no gender label: %s' % [cls2spk.get(c) for c in ids if id_gender[c] == '?'])

In [ ]:
# Cell 4 - the FOP model, its loss, and the EER metric
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve

class Branch(nn.Module):
    # one modality: Linear -> BatchNorm -> ReLU -> Dropout -> unit length
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)

class FOP(nn.Module):
    # Fusion and Orthogonal Projection (Saeed et al., ICASSP 2022) - same layout as official code
    def __init__(self, n_class, d=128):
        super().__init__()
        self.face, self.voice = Branch(4096, d), Branch(192, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused)

def opl_loss(feat, lab):
    # orthogonal projection loss: same person -> similar, different people -> orthogonal
    feat = Fn.normalize(feat, dim=1)
    dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device)
    neg = 1 - same
    pos_mean = (pos * dot).sum() / (pos.sum() + 1e-6)
    neg_mean = (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
    return (1 - pos_mean) + 0.7 * neg_mean

def eer(labels, scores):
    # scores: HIGHER = more likely same person
    fpr, tpr, _ = roc_curve(labels, scores)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return (fpr[i] + fnr[i]) / 2 * 100

XF = torch.tensor(Xf, device=dev)
XV = torch.tensor(Xv, device=dev)

@torch.no_grad()
def distance(model, xf, xv):
    # official scoring: L2 distance between face and voice embeddings. LOWER = same person
    model.eval()
    ft, vt, _, _ = model.embed(xf, xv)
    return (ft - vt).norm(dim=1).cpu().numpy()

In [ ]:
# Cell 5 - honest validation: people-disjoint folds + labelled pairs in both conditions
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    # returns (voice_rows, face_rows, labels); half positives, half negatives
    rng = np.random.default_rng(seed)
    spk = np.unique(y[idx])
    rows = {s: idx[y[idx] == s] for s in spk}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk)
        V.append(rng.choice(rows[s])); F.append(rng.choice(rows[s])); L.append(1)
        s = rng.choice(spk)
        cand = [t for t in spk if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])]
        if not cand:
            cand = [t for t in spk if t != s]
        t = rng.choice(cand)
        V.append(rng.choice(rows[s])); F.append(rng.choice(rows[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)

def val_eers(model, vsets):
    out = []
    for V, F, L in vsets:
        d = distance(model, XF[F], XV[V])
        out.append(eer(L, -d))
    return out   # [standard, gender-constrained]

# 5 folds of 14 people, balanced by gender
K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]
    rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
for k in range(K):
    print('fold %d: %d people (%s)' % (k, len(folds[k]), ''.join(sorted(id_gender[c] for c in folds[k]))))

In [ ]:
# Cell 6 - training function (official FOP settings: lr 1e-5, batch 128, alpha 2, up to 200 epochs)
import copy

def train_fop(tr_idx, vsets=None, max_epochs=200, patience=20, seed=1, lr=1e-5, alpha=2.0, bs=128):
    torch.manual_seed(seed); np.random.seed(seed)
    classes = np.unique(y[tr_idx])
    remap = {c: i for i, c in enumerate(classes)}
    ytr = torch.tensor([remap[c] for c in y[tr_idx]], device=dev)
    tr = torch.tensor(tr_idx, device=dev)
    model = FOP(len(classes)).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.01)
    best, best_ep, best_state, wait, hist = 1e9, 0, None, 0, []
    for ep in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(len(tr), device=dev)
        for b in range(0, len(tr) - bs + 1, bs):
            j = perm[b:b + bs]
            fused, logits = model(XF[tr[j]], XV[tr[j]])
            loss = Fn.cross_entropy(logits, ytr[j]) + alpha * opl_loss(fused, ytr[j])
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        e_std, e_gen = val_eers(model, vsets)
        score = (e_std + e_gen) / 2
        hist.append((ep, e_std, e_gen))
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, hist

In [ ]:
# Cell 7 - 5-fold cross-validation (this is our trustworthy local score)
import matplotlib.pyplot as plt
t0 = time.time()
res = []
plt.figure(figsize=(8, 3.5))
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]
    tr_idx = np.where([c not in val_ids for c in y])[0]
    vsets = [make_pairs(va_idx, same_gender_neg=False, seed=k),
             make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]
    model, best_ep, hist = train_fop(tr_idx, vsets)
    e_std, e_gen = val_eers(model, vsets)
    res.append((k, best_ep, e_std, e_gen))
    print('fold %d: best epoch %3d | standard EER %.2f | same-gender EER %.2f   (%.0f s)'
          % (k, best_ep, e_std, e_gen, time.time() - t0))
    h = np.array(hist)
    plt.plot(h[:, 0], (h[:, 1] + h[:, 2]) / 2, label='fold %d' % k)
plt.xlabel('epoch'); plt.ylabel('val EER (mean of 2 conditions)'); plt.legend(); plt.grid(alpha=.3); plt.show()

R = np.array(res)
CV_STD, CV_GEN = R[:, 2].mean(), R[:, 3].mean()
FINAL_EPOCHS = int(np.median(R[:, 1]))
note('CV standard EER     : %.2f +- %.2f' % (CV_STD, R[:, 2].std()))
note('CV same-gender EER  : %.2f +- %.2f' % (CV_GEN, R[:, 3].std()))
note('CV overall (mean)   : %.2f' % ((CV_STD + CV_GEN) / 2))
note('Best epochs per fold: %s -> final model trains %d epochs' % (list(R[:, 1].astype(int)), FINAL_EPOCHS))

In [ ]:
# Cell 8 - final model on all 70 people, score the DEV pairs, build the submission ZIP
all_idx = np.arange(len(y))
final, _, _ = train_fop(all_idx, vsets=None, max_epochs=FINAL_EPOCHS)

RUN = '02_fop_baseline'
OUT = '/content/sub_' + RUN
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang, fname in LANG_FILE.items():
        key = '/' + track + '/'
        pair_ids = [l.split()[0] for l in open(find_one(key, lang + '_test.txt')) if l.strip()]
        df_, _ = read_feat(find_one(key, lang + '_test_faces.csv'), 4096)
        dv_, _ = read_feat(find_one(key, lang + '_test_voices.csv'), 192)
        assert len(pair_ids) == len(df_) == len(dv_)
        d = distance(final, torch.tensor(df_, device=dev), torch.tensor(dv_, device=dev))
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, s in zip(pair_ids, d):
                f.write('%s %.6f\n' % (pid, s))   # distance: LOWER = same person (as official code)
        print('%-9s %-7s %d scores written' % (track, lang, len(d)))

SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
ZIP_PATH = os.path.join(SUBDIR, RUN + '.zip')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
print('\nZIP contents:', zipfile.ZipFile(ZIP_PATH).namelist())
note('Submission ZIP saved: ' + ZIP_PATH[len(BASE):])

MODELDIR = os.path.join(PROJECT, 'models'); os.makedirs(MODELDIR, exist_ok=True)
torch.save(final.state_dict(), os.path.join(MODELDIR, RUN + '.pt'))

In [ ]:
# Cell 9 - log the experiment + summary
LOG = os.path.join(PROJECT, 'experiments.csv')
row = pd.DataFrame([{'date': time.strftime('%Y-%m-%d %H:%M'), 'run': RUN,
                     'change': 'official FOP re-implementation (VGGFace+ECAPA features)',
                     'seed': 1, 'cv_std': round(CV_STD, 2), 'cv_gender': round(CV_GEN, 2),
                     'cv_overall': round((CV_STD + CV_GEN) / 2, 2), 'codabench_dev': ''}])
row.to_csv(LOG, mode='a', header=not os.path.exists(LOG), index=False)

print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)